# Improving Our Text Features

Last week, we built our first language model using a Bag of Words approach. That was a great start, we cleaned the text, counted words, and trained a classifier. But Bag of Words has a few major limitations:

> If your documents are different lengths, longer documents will naturally have higher word counts, even for words that aren't particularly important.

This can cause a bias in the model, where longer documents dominate simply because of their size.

### The Fix: Term Frequency

To address this, we can normalize word counts by the total number of words in each document. This gives us the **term frequency**:

$$
\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}
$$


This transformation turns raw counts into **proportions**, allowing us to compare documents of varying lengths more fairly.


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# Example corpus
docs = [
    "the quick brown fox",
    "the quick fox jumps",
    "the fox"
]

# Get bag of words
vectorizer = CountVectorizer()
X_counts = vectorizer.fit_transform(docs)

# Convert to tf
counts_array = X_counts.toarray()
tf_array = counts_array / counts_array.sum(axis=1, keepdims=True)

# Results
print("Vocabulary:", vectorizer.get_feature_names_out())
print("Raw Counts:\n", counts_array)
print("Term Frequency (TF):\n", tf_array)


Vocabulary: ['brown' 'fox' 'jumps' 'quick' 'the']
Raw Counts:
 [[1 1 0 1 1]
 [0 1 1 1 1]
 [0 1 0 0 1]]
Term Frequency (TF):
 [[0.25 0.25 0.   0.25 0.25]
 [0.   0.25 0.25 0.25 0.25]
 [0.   0.5  0.   0.   0.5 ]]


## Inverse Document Frequency (IDF)

Term frequency solves one problem, but it introduces another.

> What if a word shows up in *every* document?

In that case, the word is likely **not helpful for distinguishing between documents**. For example, in a collection of *Star Trek* scripts, the term *"Starfleet"* might appear in every script, so it doesn’t help us tell one episode from another.

This is similar to the idea of **zero variance features** in traditional machine learning: if a feature doesn’t change across samples, it doesn’t help the model learn.

### The Fix: Downweight Common Terms

To solve this, we introduce **inverse document frequency** (IDF), which downweights terms that appear in many documents:

$$
\text{IDF}(t) = \log\left(\frac{N}{1 + \text{df}(t)}\right)
$$

Where:  
- $N$ is the total number of documents  
- $\text{df}(t)$ is the number of documents containing term $t$

<!-- ![Inverse Document Frequency vs Document Frequency](./img/img_idf.png) -->
<img src="./img/img_idf.png" alt="IDF Plot" width="600"/>


## TF-IDF: The Best of Both Worlds

By combining term frequency and inverse document frequency, we get **TF-IDF**:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$


This approach helps us:
- Emphasize words that are **important within a document**
- Downweight words that are **common across all documents**

### Why It Matters

TF-IDF is one of the most widely used techniques in NLP for:
- Document classification
- Information retrieval
- Search engines
- Feature engineering

And the best part? `sklearn` has a ready to use `TfidfVectorizer` that handles all of this for you. We'll explore it in the next section.


In [8]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import numpy as np

# Example Corpus
docs = [
    "the quick brown fox",
    "the quick fox jumps over the lazy dog",
    "the fox"
]

# Bag of words
vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(docs)
print("Bag of Words:\n", bow.toarray())

# TF-IDF
tfidf = TfidfTransformer()
tfidf_matrix = tfidf.fit_transform(bow)
print("\nTF-IDF:\n", np.round(tfidf_matrix.toarray(), 2))

# Feature Names
print("\nVocabulary:\n", vectorizer.get_feature_names_out())


Bag of Words:
 [[1 0 1 0 0 0 1 1]
 [0 1 1 1 1 1 1 2]
 [0 0 1 0 0 0 0 1]]

TF-IDF:
 [[0.66 0.   0.39 0.   0.   0.   0.5  0.39]
 [0.   0.4  0.23 0.4  0.4  0.4  0.3  0.47]
 [0.   0.   0.71 0.   0.   0.   0.   0.71]]

Vocabulary:
 ['brown' 'dog' 'fox' 'jumps' 'lazy' 'over' 'quick' 'the']


## Spacy: Smarter NLP Tools

So far, we’ve been working with `sklearn` for model training. Now let’s add a library designed specifically for language. We'll use **spacy**, a fast and powerful NLP toolkit that provides:

- Tokenization
- Lemmatization
- POS tagging
- Named entity recognition

### Download a Language Model

Spacy models are separate from the library itself. We'll start with the small English model. Choose one:

- `en_core_web_sm`: small (~15MB) — fast, basic accuracy
- `en_core_web_md`: medium (~40MB) — includes word vectors
- `en_core_web_lg`: large (~750MB) — best accuracy, full vectors

In [9]:
# uncomment and run this once to install spacy and specified model

# !pip install spacy
# !python -m spacy download en_core_web_sm

In [10]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Captain Janeway negotiated peace with the Romulans.")
print([token.text for token in doc])

['Captain', 'Janeway', 'negotiated', 'peace', 'with', 'the', 'Romulans', '.']


## Stemming and Lemmatization

So far, we’ve transformed our raw text using methods like stopword removal and vectorization. But there’s **one more important transformation** we can apply to improve our model’s understanding of language:

> What if we could **combine different forms of the same word** into a single, meaningful token?

### The Problem

Words like:

- `"play"`, `"playing"`, `"played"`, `"plays"`

All refer to the **same base idea**, the action of "play", but they would be treated as **separate tokens** in a bag of words or TF-IDF model. That’s not ideal. We want our model to generalize the *concept*, not just memorize word forms. This is where **stemming** and **lemmatization** come in.

### Stemming

**Stemming** is a fast and simple technique that chops off word endings to reduce them to a **base form** (called a “stem”).

- It doesn't consider context
- It often produces non-real words
- But it’s fast and memory-efficient

Example:

| Word     | Stemmed |
|----------|---------|
| playing  | play    |
| played   | play    |
| happily  | happi   |
| studies  | studi   |
> Notice how `"studies"` becomes `"studi"`, not a real word, but a consistent stem.

### Lemmatization

**Lemmatization** is more sophisticated. It uses **context** and **part of speech tagging** to return the **dictionary form** of a word, known as its **lemma**.

- More accurate
- Produces real words
- Slower and more resource-intensive

Example:

| Word     | Lemma   |
|----------|---------|
| playing  | play    |
| better   | good    |
| studies  | study   |
| mice     | mouse   |
> Unlike stemming, lemmatization can convert `"better"` to `"good"` by understanding its **semantic role**.

### Which Should You Use?

If you're doing serious NLP, especially anything involving **meaning or interpretation**, use **lemmatization**.

> **Note:** spacy does not support stemming, it uses **lemmatization only**, because it’s more accurate and produces real words. For comparison, stemmers like PorterStemmer are available in NLTK, but we won’t use them in this course.

In [11]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "playing played plays better happily studies mice"
doc = nlp(text)

print("Spacy Lemmatization:")
for token in doc:
    print(f"{token.text}\t - {token.lemma_}")


Spacy Lemmatization:
playing	 - playing
played	 - play
plays	 - play
better	 - well
happily	 - happily
studies	 - study
mice	 - mouse


## Pipelines

When working with text models, we often need to apply **several preprocessing steps** before feeding the data into a classifier things like tokenization, vectorization, and normalization.

You’ll also need to repeat these steps **exactly the same way** any time you want to make a prediction on new data. That’s where things can get tricky:  
If you forget a step, or apply transformations inconsistently, it can introduce **subtle and hard to find bugs** in your model pipeline.

### `sklearn.pipeline.Pipeline`

Sklearn provides a helpful utility called a **Pipeline**, which allows you to **bundle multiple steps together** into a single, reusable model object. With a pipeline, you can:
- Cleanly organize your workflow
- Ensure your transformations are always applied in the correct order
- Train and predict using a single `.fit()` or `.predict()` call

### How Pipelines Work

A pipeline chains together multiple estimators:
- Each step (except the last) must be a **transformer** (i.e., it must implement `.fit_transform()` or `.transform()`)
- The **final step** can be any estimator, usually a classifier or regressor

The data flows from step to step:


In [12]:
import spacy
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

nlp = spacy.load("en_core_web_sm")

def spacy_tokenizer(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_punct and not token.is_space]

# Sample corpus
texts = [
    "I love this movie, it was fantastic and exciting!",
    "What a terrible film. I hated every moment.",
    "Absolutely wonderful acting and a great story.",
    "The plot was boring and predictable.",
    "A masterpiece. Beautifully shot and emotionally powerful.",
    "Worst movie I have seen this year.",
    "An outstanding performance by the lead actor.",
    "I walked out halfway — it was awful.",
    "Brilliant direction and stunning visuals.",
    "Completely unwatchable. Don't waste your time."
]

# Labels: 1 = positive, 0 = negative
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.3, random_state=42)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(tokenizer=spacy_tokenizer, lowercase=True)),
    ('clf', LogisticRegression())
])

# Train the model
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print(y_pred)


[1 1 1]


/home/pantherson/.virtualenvs/nlp-venv/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


# Summary

This week we focused on improving our text models by making our features more meaningful, flexible, and consistent.

## TF-IDF
- Combines TF and IDF
  - TF normalizes word counts within a document
  - IDF downweights common words across all documents
- Gives higher weight to important, uncommon words
- Implemented easily with `TfidfVectorizer`

## Spacy
- Python NLP library for working with language data
- Tools we used:
  - Tokenization
  - Lemmatization
  - Custom tokenization inside `TfidfVectorizer`

## Pipelines
- Sklearn tool to bundle preprocessing and modeling steps
- Helps keep your code:
  - Clean
  - Repeatable
  - Safe for making predictions on new data